In [1]:
# ============================================================
# ENERGY INTELLIGENCE HUB – EXPORT ALL REPORTS
# Generates ALL Excel reports for the project
# ============================================================

import psycopg2
import pandas as pd
import os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from datetime import datetime

# ============================================================
# 1. CONNECTION
# ============================================================
conn = psycopg2.connect(
    host="localhost",
    database="energy_intelligence_project1",
    user="postgres",
    password="Solovigo123DELLYS"  
)

# ============================================================
# 2. CREATE OUTPUTS FOLDER (IF NOT EXISTS)
# ============================================================
os.makedirs('../outputs', exist_ok=True)

print("="*60)
print("📊 GENERATING ENERGY REPORTS")
print("="*60)

# ============================================================
# 3. REPORT 1: LOAD FACTOR (Efficiency Ranking)
# ============================================================
print("\n📊 Report 1: Load Factor Ranking...")

query_load_factor = """
SELECT 
    b.building_name,
    ROUND(SUM(er.energy_kwh), 2) AS total_kwh,
    ROUND(MAX(er.demand_kw), 2) AS peak_demand,
    ROUND(AVG(er.demand_kw), 2) AS avg_demand,
    ROUND((SUM(er.energy_kwh) / (MAX(er.demand_kw) * (COUNT(*) * 0.25))) * 100, 2) AS load_factor_percent,
    CASE 
        WHEN (SUM(er.energy_kwh) / (MAX(er.demand_kw) * (COUNT(*) * 0.25))) > 0.6 THEN 'Efficient'
        WHEN (SUM(er.energy_kwh) / (MAX(er.demand_kw) * (COUNT(*) * 0.25))) > 0.4 THEN 'Average'
        ELSE 'Inefficient'
    END AS efficiency_rating
FROM energy_readings er
JOIN buildings b ON er.building_id = b.building_id
GROUP BY b.building_name
ORDER BY load_factor_percent ASC;
"""
df_load_factor = pd.read_sql(query_load_factor, conn)

# ============================================================
# 4. REPORT 2: PEAK DEMAND (Top 10 Spikes)
# ============================================================
print("\n📊 Report 2: Peak Demand (Top 10)...")

query_peak_demand = """
SELECT 
    b.building_name,
    pd.peak_date,
    pd.demand_kw,
    pd.demand_kw * 15.00 AS demand_charge_eur
FROM peak_demand pd
JOIN buildings b ON pd.building_id = b.building_id
ORDER BY demand_charge_eur DESC
LIMIT 10;
"""
df_peak_demand = pd.read_sql(query_peak_demand, conn)

# ============================================================
# 5. REPORT 3: ANOMALY SUMMARY
# ============================================================
print("\n📊 Report 3: Anomaly Summary...")

query_anomalies = """
SELECT 
    b.building_name,
    a.anomaly_type,
    a.severity,
    a.description,
    a.cost_impact_eur,
    a.status,
    a.detected_at
FROM anomalies a
JOIN buildings b ON a.building_id = b.building_id
ORDER BY a.cost_impact_eur DESC;
"""
df_anomalies = pd.read_sql(query_anomalies, conn)

# ============================================================
# 6. REPORT 4: ANNUAL DEMAND CHARGES
# ============================================================
print("\n📊 Report 4: Annual Demand Charges...")

query_annual_demand = """
WITH monthly_peaks AS (
    SELECT 
        building_id,
        EXTRACT(YEAR FROM peak_date) AS year,
        DATE_TRUNC('month', peak_date) AS month,
        MAX(demand_kw) AS monthly_max_demand_kw
    FROM peak_demand
    GROUP BY building_id, EXTRACT(YEAR FROM peak_date), DATE_TRUNC('month', peak_date)
)
SELECT 
    b.building_name,
    mp.year,
    ROUND(SUM(mp.monthly_max_demand_kw * 15.00), 2) AS annual_demand_charge_eur
FROM monthly_peaks mp
JOIN buildings b ON mp.building_id = b.building_id
GROUP BY b.building_name, mp.year
ORDER BY annual_demand_charge_eur DESC;
"""
df_annual_demand = pd.read_sql(query_annual_demand, conn)

# ============================================================
# 7. REPORT 5: TOTAL ANNUAL BILL (Energy + Demand + Fixed)
# ============================================================
print("\n📊 Report 5: Total Annual Bill...")

query_total_bill = """
WITH annual_energy AS (
    SELECT 
        b.building_id,
        b.building_name,
        EXTRACT(YEAR FROM er.timestamp) AS year,
        ROUND(SUM(er.energy_kwh) * 0.12, 2) AS energy_cost_eur
    FROM energy_readings er
    JOIN buildings b ON er.building_id = b.building_id
    GROUP BY b.building_id, b.building_name, EXTRACT(YEAR FROM er.timestamp)
),
monthly_peaks AS (
    SELECT 
        building_id,
        EXTRACT(YEAR FROM peak_date) AS year,
        DATE_TRUNC('month', peak_date) AS month,
        MAX(demand_kw) AS monthly_max_demand_kw
    FROM peak_demand
    GROUP BY building_id, EXTRACT(YEAR FROM peak_date), DATE_TRUNC('month', peak_date)
),
annual_demand AS (
    SELECT 
        building_id,
        year,
        ROUND(SUM(monthly_max_demand_kw * 15.00), 2) AS demand_charge_eur
    FROM monthly_peaks
    GROUP BY building_id, year
)
SELECT 
    ae.building_name,
    ae.year,
    ae.energy_cost_eur,
    ad.demand_charge_eur,
    912.50 AS fixed_fees_eur,
    ROUND(ae.energy_cost_eur + ad.demand_charge_eur + 912.50, 2) AS total_annual_bill_eur
FROM annual_energy ae
JOIN annual_demand ad ON ae.building_id = ad.building_id AND ae.year = ad.year
ORDER BY total_annual_bill_eur DESC;
"""
df_total_bill = pd.read_sql(query_total_bill, conn)

# ============================================================
# 8. REPORT 6: FORECAST SAMPLE (48 hours)
# ============================================================
print("\n📊 Report 6: Forecast Sample (48h)...")

query_forecast = """
SELECT 
    forecast_timestamp,
    forecast_energy_kwh,
    forecast_demand_kw,
    model_name
FROM energy_forecasts
WHERE building_id = 17
ORDER BY forecast_timestamp
LIMIT 48;
"""
df_forecast = pd.read_sql(query_forecast, conn)

# ============================================================
# 9. REPORT 7: DEMAND CHARGE PERCENTAGE
# ============================================================
print("\n📊 Report 7: Demand Charge Percentage...")

query_demand_percentage = """
WITH annual_energy AS (
    SELECT 
        b.building_id,
        b.building_name,
        EXTRACT(YEAR FROM er.timestamp) AS year,
        ROUND(SUM(er.energy_kwh) * 0.12, 2) AS energy_cost_eur
    FROM energy_readings er
    JOIN buildings b ON er.building_id = b.building_id
    GROUP BY b.building_id, b.building_name, EXTRACT(YEAR FROM er.timestamp)
),
monthly_peaks AS (
    SELECT 
        building_id,
        EXTRACT(YEAR FROM peak_date) AS year,
        DATE_TRUNC('month', peak_date) AS month,
        MAX(demand_kw) AS monthly_max_demand_kw
    FROM peak_demand
    GROUP BY building_id, EXTRACT(YEAR FROM peak_date), DATE_TRUNC('month', peak_date)
),
annual_demand AS (
    SELECT 
        building_id,
        year,
        ROUND(SUM(monthly_max_demand_kw * 15.00), 2) AS demand_charge_eur
    FROM monthly_peaks
    GROUP BY building_id, year
)
SELECT 
    ae.building_name,
    ae.year,
    ae.energy_cost_eur,
    ad.demand_charge_eur,
    ROUND(ae.energy_cost_eur + ad.demand_charge_eur + 912.50, 2) AS total_bill_eur,
    ROUND((ad.demand_charge_eur / (ae.energy_cost_eur + ad.demand_charge_eur + 912.50)) * 100, 2) AS demand_percentage
FROM annual_energy ae
JOIN annual_demand ad ON ae.building_id = ad.building_id AND ae.year = ad.year
ORDER BY demand_percentage DESC;
"""
df_demand_percentage = pd.read_sql(query_demand_percentage, conn)

# ============================================================
# 10. EXPORT ALL TO EXCEL WITH FORMATTING
# ============================================================
print("\n💾 Exporting all reports to Excel...")

output_path = '../outputs/energy_intelligence_reports.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    
    # Write all dataframes to separate sheets
    df_load_factor.to_excel(writer, sheet_name='Load Factor Ranking', index=False)
    df_peak_demand.to_excel(writer, sheet_name='Peak Demand (Top 10)', index=False)
    df_anomalies.to_excel(writer, sheet_name='Anomaly Summary', index=False)
    df_annual_demand.to_excel(writer, sheet_name='Annual Demand Charges', index=False)
    df_total_bill.to_excel(writer, sheet_name='Total Annual Bill', index=False)
    df_forecast.to_excel(writer, sheet_name='Forecast Sample (48h)', index=False)
    df_demand_percentage.to_excel(writer, sheet_name='Demand Charge %', index=False)

print(f"✅ All reports saved to: {output_path}")
print("📁 Files saved in: outputs/energy_intelligence_reports.xlsx")

# ============================================================
# 11. CLOSE CONNECTION
# ============================================================
conn.close()

print("\n" + "="*60)
print("🎉 ALL REPORTS GENERATED SUCCESSFULLY!")
print("="*60)
print("📊 Reports included:")
print("   1. Load Factor Ranking")
print("   2. Peak Demand (Top 10)")
print("   3. Anomaly Summary")
print("   4. Annual Demand Charges")
print("   5. Total Annual Bill")
print("   6. Forecast Sample (48h)")
print("   7. Demand Charge Percentage")

📊 GENERATING ENERGY REPORTS

📊 Report 1: Load Factor Ranking...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\2074081627.py:54: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_load_factor = pd.read_sql(query_load_factor, conn)



📊 Report 2: Peak Demand (Top 10)...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\2074081627.py:72: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_peak_demand = pd.read_sql(query_peak_demand, conn)



📊 Report 3: Anomaly Summary...

📊 Report 4: Annual Demand Charges...


...


📊 Report 5: Total Annual Bill...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\2074081627.py:164: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_total_bill = pd.read_sql(query_total_bill, conn)



📊 Report 6: Forecast Sample (48h)...

📊 Report 7: Demand Charge Percentage...


C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\2074081627.py:182: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_forecast = pd.read_sql(query_forecast, conn)
C:\Users\DeyStore\AppData\Local\Temp\ipykernel_14992\2074081627.py:228: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_demand_percentage = pd.read_sql(query_demand_percentage, conn)



💾 Exporting all reports to Excel...
✅ All reports saved to: ../outputs/energy_intelligence_reports.xlsx
📁 Files saved in: outputs/energy_intelligence_reports.xlsx

🎉 ALL REPORTS GENERATED SUCCESSFULLY!
📊 Reports included:
   1. Load Factor Ranking
   2. Peak Demand (Top 10)
   3. Anomaly Summary
   4. Annual Demand Charges
   5. Total Annual Bill
   6. Forecast Sample (48h)
   7. Demand Charge Percentage
